# NB07 — CARNIVAL (**S3 demoted to presentation**)

**In:** DepMap breast cell-line expression, OmniPath signalling PPI
**Out:** `data/interim/causal_networks/{sample_id}.json` (explanation layer)
**Gates:** (1) vs CRISPR essentiality AUROC ≥ 0.65 — recorded failure, threshold not revised.
(2) vs GDSC target sensitivity Spearman — re-validation of what S4 actually needs.

ODE initial conditions come from PROGENy/CollecTRI, not from the ILP.
Essentiality and signalling activity are different biology; a chance essentiality
AUROC is a valid negative if networks vary across lines (mean Jaccard < 0.8).


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe
try:
    import certifi, os
    os.environ.setdefault("SSL_CERT_FILE", certifi.where())
    os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
except Exception:
    pass

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures"
for d in (RAW, INTERIM, REF, ARTIFACTS, FIGURES, INTERIM / "causal_networks"):
    d.mkdir(parents=True, exist_ok=True)

# Laptop vs VPS. Smoke passes are provisional until a full run converts them.
# NB01 and NB04 stay full: harmonisation and the VAE are cheap.
SMOKE_TEST = False
N_SAMPLES  = None   # full TCGA-BRCA; do not cap
N_SC_CELLS = 25_000  # Wu reference subsample if RAM is tight
N_PATIENTS = None
N_DRUGS    = None

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    if "sample_ids" not in kwargs:
        kwargs.setdefault("cohort", False)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
# Config
# CARNIVAL is throughput, not memory: one ILP is 1–2 GB / 30–90 s.
# Smoke: ~50 solves. Full: every sample (VPS wall-clock, embarrassingly parallel).
AUROC_MIN = 0.65          # do not revise; essentiality fail stands
JACCARD_MAX = 0.8         # mean pairwise Jaccard; ≥ this ⇒ networks are copies
GDSC_RHO_MIN = 0.3        # target-sensitivity re-validation (what S4 needs)
TIMELIMIT = 90 if SMOKE_TEST else 300
MAX_SAMPLES = N_PATIENTS
R_SCRIPT = V2_ROOT / "notebooks" / "r" / "run_carnival.R"
NET_DIR = INTERIM / "causal_networks"
NET_DIR.mkdir(exist_ok=True)
import subprocess, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from pathlib import Path


In [ ]:
# Load — TF activity from DepMap breast lines, then OmniPath PPI.
# If ACH jsons already exist, skip decoupler + ILP rebuild.
from carnival_pkn import build_carnival_pkn, load_omnipath_ppi
from io_data import load_depmap_breast_expression
ode_nodes = pd.read_csv(REF / "ode_nodes.csv")["gene"].astype(str).tolist() if (REF / "ode_nodes.csv").exists() else []
expr_p = REPO_ROOT / "depmap_data" / "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv"
model_p = REPO_ROOT / "depmap_data" / "Model.csv"
pkn_p = INTERIM / "pkn_carnival_symbols.parquet"
depmap = REPO_ROOT / "depmap_data" / "CRISPRGeneEffect.csv"
existing = [p for p in NET_DIR.glob("*.json") if p.stem.startswith("ACH-")]
tf = None
pkn = pd.read_parquet(pkn_p) if pkn_p.exists() else None
if existing and pkn is not None:
    print("reusing", len(existing), "CARNIVAL jsons; skip TF/PKN rebuild")
elif expr_p.exists() and model_p.exists():
    mat = load_depmap_breast_expression(expr_p, model_p, n=MAX_SAMPLES)
    print("DepMap breast expression", mat.shape, "available", mat.attrs.get("n_breast_available"))
    try:
        import decoupler as dc
        tf_net = dc.op.collectri(organism="human")
        tf_res = dc.mt.ulm(mat, tf_net)
        tf = pd.DataFrame(tf_res[0] if isinstance(tf_res, tuple) else tf_res)
        if list(tf.index) != list(mat.index) and list(tf.columns) == list(mat.index):
            tf = tf.T
        tf.index = mat.index.astype(str)
        tf = tf.replace([np.inf, -np.inf], np.nan)
        print("cell-line TF activity", tf.shape)
    except Exception as e:
        print("decoupler on DepMap failed", e)
        tf = mat.iloc[:, :40]
        tf.columns = [str(c) for c in tf.columns]
    raw = None
    try:
        raw = load_omnipath_ppi(INTERIM / "omnipath_ppi.parquet")
        print("OmniPath PPI cache", raw.shape)
    except Exception as e:
        print("OmniPath PPI unavailable, literature edges only", e)
    pkn, meta = build_carnival_pkn(list(map(str, tf.columns)), ode_nodes, raw=raw, max_edges=5000)
    pkn.to_parquet(pkn_p, index=False)
    print("CARNIVAL PKN", meta)
else:
    print("DepMap expression missing")


In [ ]:
# Compute — reuse solved networks; do not re-run the ILP if ACH jsons are present
n_real = 0
first_err = None
existing = [p for p in NET_DIR.glob("*.json") if p.stem.startswith("ACH-")]
if existing:
    n_real = 0
    for p in existing:
        obj = json.loads(p.read_text())
        if obj.get("mode") != "fallback_threshold" and "error" not in (obj.get("result") or {}):
            n_real += 1
    print("reusing CARNIVAL jsons", len(existing), "real", n_real)
elif tf is not None and pkn is not None:
    for p in NET_DIR.glob("*.json"):
        p.unlink()
    tf = tf.copy()
    tf.index = tf.index.astype(str)
    tf.columns = tf.columns.astype(str)
    pkn_nodes = set(pkn["source"].astype(str)) | set(pkn["target"].astype(str))
    samples = list(tf.index)
    if MAX_SAMPLES is not None:
        samples = samples[: int(MAX_SAMPLES)]
    pkn.to_parquet(pkn_p, index=False)
    for sid in samples:
        vals = tf.loc[sid].replace([np.inf, -np.inf], np.nan).dropna()
        vals.index = vals.index.astype(str)
        vals = vals[vals.index.isin(pkn_nodes)]
        if len(vals) < 5:
            print("skip", sid, "TFs in PKN", len(vals))
            continue
        top = vals.abs().nlargest(min(25, len(vals)))
        meas = vals.loc[top.index].to_frame().T
        meas.to_parquet(INTERIM / "_tf_row.parquet", index=False)
        cmd = ["Rscript", str(R_SCRIPT), "--pkn", str(pkn_p), "--tf", str(INTERIM / "_tf_row.parquet"),
               "--outdir", str(NET_DIR), "--sample_id", sid, "--timelimit", str(TIMELIMIT)]
        rc = subprocess.run(cmd, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
        outp = NET_DIR / f"{sid}.json"
        obj = json.loads(outp.read_text()) if outp.exists() else {}
        res = obj.get("result") if isinstance(obj.get("result"), dict) else {}
        real = bool(res) and "error" not in res and ("nodesAttributes" in res or "weightedSIF" in res)
        if real:
            n_real += 1
            print(sid, "InvCARNIVAL", obj.get("elapsed_sec"), flush=True)
            continue
        err = res.get("error") if res else (rc.stderr[-500:] if rc.stderr else "no CARNIVAL json")
        if first_err is None:
            first_err = err
            print("first CARNIVAL error", sid, err)
        thr = float(meas.iloc[0].abs().median())
        active = [str(g) for g, v in meas.iloc[0].items() if abs(float(v)) >= thr]
        edges = pkn[pkn["source"].isin(active) | pkn["target"].isin(active)].head(50).to_dict("records")
        outp.write_text(json.dumps({
            "sample_id": sid, "mode": "fallback_threshold", "timed_out": False,
            "active_nodes": active, "edges": edges, "r_returncode": rc.returncode,
            "carnival_error": err,
        }, indent=2, default=str))
    print("networks", len(list(NET_DIR.glob('*.json'))), "real_carnival", n_real, "first_err", first_err)


In [ ]:
# GATE
auroc, note = 0.0, "no networks"
nets = list(NET_DIR.glob("*.json"))
n_real = sum(1 for n in nets if json.loads(n.read_text()).get("mode") != "fallback_threshold")
if nets and depmap.exists() and n_real >= 10:
    ge = pd.read_csv(depmap, index_col=0)
    ge.columns = ge.columns.str.replace(r" \(\d+\)$", "", regex=True)
    line_ids = []
    for n in nets:
        obj = json.loads(n.read_text())
        if obj.get("mode") != "fallback_threshold":
            line_ids.append(str(obj.get("sample_id") or n.stem))
    sub = ge.loc[ge.index.intersection(line_ids)]
    if sub.empty:
        sub = ge.loc[ge.index.intersection(
            pd.read_csv(REPO_ROOT / "depmap_data" / "Model.csv").loc[
                lambda m: m["OncotreeLineage"].astype(str).str.contains("Breast", case=False, na=False), "ModelID"
            ]
        )]
    essential = (sub.median() < -0.5).astype(int)
    pkn_nodes = set(pkn["source"].astype(str)) | set(pkn["target"].astype(str)) if pkn is not None else set(essential.index)

    def carnival_active(obj):
        if obj.get("mode") == "fallback_threshold":
            return []
        res = obj.get("result") or {}
        na = res.get("nodesAttributes")
        rows = []
        if isinstance(na, list):
            rows = [r for r in na if isinstance(r, dict)]
        elif isinstance(na, dict) and na.get("Node") is not None:
            nodes = na.get("Node")
            if not isinstance(nodes, list):
                nodes = [nodes]
            acts = na.get("AvgAct", na.get("Activity", na.get("activity", [0] * len(nodes))))
            if not isinstance(acts, list):
                acts = [acts]
            rows = [{"Node": n, "AvgAct": a} for n, a in zip(nodes, acts)]
        names = []
        for row in rows:
            node = row.get("Node") or row.get("node")
            if node in ("Perturbation", "INPUT"):
                continue
            act = row.get("AvgAct", row.get("Activity", row.get("activity", 0)))
            try:
                if node is not None and float(act or 0) != 0:
                    names.append(str(node))
            except (TypeError, ValueError):
                continue
        return names

    def score_universe(universe):
        scores = {g: 0.0 for g in universe}
        for n in nets:
            obj = json.loads(n.read_text())
            if obj.get("mode") == "fallback_threshold":
                continue
            for g in carnival_active(obj):
                for cand in (g, g.replace("_", "-")):
                    if cand in scores:
                        scores[cand] += 1
        y = essential.reindex(universe).dropna()
        s = pd.Series(scores).reindex(y.index).fillna(0)
        if y.nunique() < 2:
            return float("nan")
        return float(roc_auc_score(y, s))

    genes_all = list(essential.index)
    genes_pkn = [g for g in genes_all if g in pkn_nodes]
    auroc_all = score_universe(genes_all)
    auroc = score_universe(genes_pkn)
    note = (f"n_lines={sub.shape[0]} n_pkn_genes={len(genes_pkn)} n_genome={len(genes_all)} "
            f"n_real_carnival={n_real} PKN_AUROC={auroc:.3f} genome_AUROC={auroc_all:.3f} "
            f"source=depmap_cell_lines")
elif not depmap.exists():
    note = "DepMap CRISPR missing"
else:
    note = f"CARNIVAL identifier/measurement join empty n_real={n_real} n_json={len(nets)}"
note = (note + " | threshold not revised (0.65); S3 demoted to presentation")
gate("NB07", "carnival_vs_depmap_essentiality", float(0.0 if auroc != auroc else auroc), AUROC_MIN,
     n=n_real, min_n=10, smoke_test=False, note=note)


In [ ]:
# Post-hoc: do networks vary, and is essentiality binarisation degenerate?
from carnival_validate import (
    active_set, activity_map, essentiality_positives_per_line,
    gdsc_target_sensitivity, load_network_dir, variation_summary,
)
nets_obj = load_network_dir(NET_DIR)
sets_all = {sid: active_set(obj) for sid, obj in nets_obj.items()}
var_all = variation_summary(sets_all)
if "ge" not in dir() or not isinstance(ge, pd.DataFrame) or ge.empty:
    ge = pd.read_csv(depmap, index_col=0) if depmap.exists() else pd.DataFrame()
    if not ge.empty:
        ge.columns = ge.columns.str.replace(r" \(\d+\)$", "", regex=True)
crispr_ids = [sid for sid in sets_all if sid in ge.index]
sets_cr = {sid: sets_all[sid] for sid in crispr_ids}
var = variation_summary(sets_cr) if sets_cr else var_all
ess = essentiality_positives_per_line(ge, crispr_ids) if crispr_ids else {"n_lines": 0, "median": float("nan"), "min": 0, "degenerate": True}
diag = {"all_lines": var_all, "crispr_lines": var, "essentiality_positives": ess}
(INTERIM / "NB07_network_variation.json").write_text(json.dumps(diag, indent=2, default=str))
print("variation CRISPR", var)
print("ess positives after pan-drop", ess)
# Informative input: mean Jaccard < 0.8 (networks are not copies).
jacc = var.get("jaccard_mean")
jacc = float(jacc) if jacc == jacc else 1.0
gate("NB07", "carnival_network_variation", jacc, JACCARD_MAX, direction="lte",
     n=int(var.get("n_networks") or 0), min_n=10, smoke_test=False,
     note=(f"size mean={var.get('size_mean'):.1f} range={var.get('size_min')}-{var.get('size_max')} "
           f"union={var.get('union')} core={var.get('core_all_lines')} singletons={var.get('singletons')} "
           f"frac_j>0.8={var.get('frac_jaccard_gt_0.8'):.3f} | ess_pos median={ess.get('median')} min={ess.get('min')} "
           f"{'INFORMATIVE' if var.get('informative') and not ess.get('degenerate') else 'UNINFORMATIVE'}"))


In [ ]:
# Re-validate vs GDSC: does inferred activity of a node predict sensitivity to inhibitors of that node?
gdsc_files = list((RAW / "gdsc2").glob("*.xlsx")) + list((RAW / "gdsc2").glob("*.csv"))
gdsc_hit = {"n_pairs": 0, "rho": float("nan"), "note": "GDSC2 file missing"}
if gdsc_files and model_p.exists() and nets_obj:
    f = gdsc_files[0]
    gdsc = pd.read_excel(f) if f.suffix == ".xlsx" else pd.read_csv(f)
    model = pd.read_csv(model_p)
    acts = {sid: activity_map(obj) for sid, obj in nets_obj.items()}
    gdsc_hit = gdsc_target_sensitivity(acts, gdsc, model)
(INTERIM / "NB07_gdsc_target.json").write_text(json.dumps(gdsc_hit, indent=2, default=str))
print("GDSC target", gdsc_hit)
rho_g = gdsc_hit.get("rho")
rho_g = float(rho_g) if rho_g == rho_g else 0.0
thin_g = int(gdsc_hit.get("n_pairs") or 0) < 20 or gdsc_hit.get("rho") != gdsc_hit.get("rho")
gate("NB07", "carnival_vs_gdsc_target_sensitivity", rho_g, GDSC_RHO_MIN,
     n=int(gdsc_hit.get("n_pairs") or 0), min_n=20, smoke_test=False,
     insufficient_data=thin_g,
     note=(f"{gdsc_hit.get('note')} n_lines={gdsc_hit.get('n_lines')} "
           f"p={gdsc_hit.get('p')} any_active_frac={gdsc_hit.get('any_active_frac')} "
           f"| S3 demoted: ODE x0 from PROGENy/CollecTRI, not CARNIVAL ILP"))


In [ ]:
print("CARNIVAL json count", len(list(NET_DIR.glob('*.json'))))
